In [1]:
from datasets import load_dataset
import pandas as pd




In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Tobi-Bueck/customer-support-tickets")
ds

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})

In [3]:
df = pd.DataFrame(ds["train"])
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [4]:
df_english = df[df["language"] == "en"]
df_english.shape

(28261, 16)

In [5]:
df_english.head()
df_english.columns

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7', 'tag_8'],
      dtype='str')

In [6]:
df_selected = df_english[["subject","body", "priority", "queue"]]
df_selected.head()

,subject,body,priority,queue
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",high,Technical Support
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",medium,Returns and Exchanges
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",low,Billing and Payments
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",medium,Sales and Pre-Sales
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",high,Technical Support


In [7]:
df_selected["body"].iloc[0]

'Dear Customer Support Team,\\n\\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\\n\\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?'

In [8]:
df_selected["priority"].unique(),df_selected["queue"].unique(),df_selected["subject"].unique()

(<ArrowStringArray>
 ['high', 'medium', 'low']
 Length: 3, dtype: str,
 <ArrowStringArray>
 [              'Technical Support',           'Returns and Exchanges',
             'Billing and Payments',             'Sales and Pre-Sales',
  'Service Outages and Maintenance',                 'Product Support',
                       'IT Support',                'Customer Service',
                  'Human Resources',                 'General Inquiry']
 Length: 10, dtype: str,
 <ArrowStringArray>
 [                                              'Account Disruption',
                'Query About Smart Home System Integration Features',
                                 'Inquiry Regarding Invoice Details',
            'Question About Marketing Agency Software Compatibility',
                                                     'Feature Query',
                                              'System Interruptions',
                 'Connectivity Problems with Printer on MacBook Pro',
              

In [9]:
import re

category_keywords = {
    "account": [
        "account", "access", "disruption", "profile", "user"
    ],
    "billing": [
        "invoice", "billing", "payment", "refund", "pricing"
    ],
    "bug": [
        "bug", "error", "issue", "problem", "inaccurate", "failure"
    ],
    "deployment": [
        "deploy", "deployment", "release", "rollout", "installation"
    ],
    "infrastructure": [
        "outage", "interrupt", "maintenance", "downtime", "system interruption"
    ],
    "integration": [
        "integration", "compatibility", "api", "connect", "smart home"
    ],
    "login": [
        "login", "sign in", "authentication", "password", "access issue"
    ],
    "network": [
        "network", "connectivity", "vpn", "printer", "wifi"
    ],
    "security": [
        "security", "breach", "data leak", "privacy", "secure"
    ]
}


In [10]:
def classify_subject(subject):
    if not isinstance(subject, str):
        return "other"   # handle NaN / non-text safely

    subject = subject.lower()

    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword in subject:
                return category

    return "other"

df_selected["category"] = df_selected["subject"].apply(classify_subject)
df_selected.head()

,subject,body,priority,queue,category
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",high,Technical Support,account
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",medium,Returns and Exchanges,integration
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",low,Billing and Payments,billing
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",medium,Sales and Pre-Sales,integration
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",high,Technical Support,other


In [11]:
for i in range(10):
    print(f"Row {i}:")
    print(df_selected["body"].iloc[i])

Row 0:
Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?
Row 1:
Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request detailed information about the capabilities of your smart home integration products listed on your website. As a potential customer aiming to develop a seamlessly interconnected home environment, it is essential to understand how your products interact with various smart home platforms.\n\nCould you kindly provide detailed compatibility info

In [12]:
df_selected["description"] = df_selected["subject"] + " " + df_selected["body"]
df_selected.head()

,subject,body,priority,queue,category,description
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",high,Technical Support,account,"Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",medium,Returns and Exchanges,integration,Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",low,Billing and Payments,billing,Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",medium,Sales and Pre-Sales,integration,Question About Marketing Agency Software Compa...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",high,Technical Support,other,"Feature Query Dear Customer Support,\n\nI hope..."


In [13]:
new_data = pd.DataFrame({"description": df_selected["description"]})
new_data.head()

,description
1,"Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...
5,"Feature Query Dear Customer Support,\n\nI hope..."


In [14]:
new_data.to_csv("customer_support_tickets.csv", index=True)

In [2]:
import pandas as pd

df = pd.read_csv("classified_tickets.csv")
df.head()

,id,description,category,priority
0,1,"Account Disruption Dear Customer Support Team,...",account,High
1,2,Query About Smart Home System Integration Feat...,security,High
2,3,Inquiry Regarding Invoice Details Dear Custome...,account,High
3,4,Question About Marketing Agency Software Compa...,security,High
4,5,"Feature Query Dear Customer Support,\n\nI hope...",security,High


In [3]:
for i in range(10):
    print(f"Row {i}:")
    print(df["description"].iloc[i])

Row 0:
Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?
Row 1:
Query About Smart Home System Integration Features Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request detailed information about the capabilities of your smart home integration products listed on your website. As a potential customer aiming to develop a seamlessly interconnected home environment, it is essential to understand how your products interact with various smart h